# 03 — Low volatility

Hold the least volatile quartile, weighted by inverse volatility.

The low-volatility anomaly — that low-risk stocks have historically earned
*higher* risk-adjusted returns than high-risk ones, contradicting a plain CAPM
reading — is among the most replicated results in equities, and one of the few
that survives transaction costs at this turnover.

Weighting by inverse volatility rather than equally pushes the portfolio further
along the same axis the selection is made on, which is the point of the strategy.

Standalone: no `portfolio_agent` import.

## Setup

In [ ]:
# Dependencies. Torch is only needed by the two learned strategies (04, 05).
# !pip install -q pandas numpy pyarrow matplotlib huggingface_hub torch

import sys, pathlib

# afa_lab.py sits next to this notebook. On Colab (or anywhere the file is
# missing) fetch it from the repo — that is the only network call that touches
# GitHub, and nothing else here imports the portfolio_agent package.
if not pathlib.Path("afa_lab.py").exists():
    import urllib.request
    URL = ("https://raw.githubusercontent.com/3dwag98/afa/main/"
           "notebooks/standalone/afa_lab.py")
    urllib.request.urlretrieve(URL, "afa_lab.py")
    print("fetched afa_lab.py")

sys.path.insert(0, ".")
import afa_lab as L

import numpy as np
import pandas as pd

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)
print("toolkit loaded | torch available:", L.TORCH_AVAILABLE)

In [ ]:
# NSE large caps. Any symbol absent from the dataset is skipped rather than
# failing the run, so this list does not have to be exactly right.
UNIVERSE = [
    "RELIANCE",
    "TCS",
    "HDFCBANK",
    "INFY",
    "ICICIBANK",
    "HINDUNILVR",
    "ITC",
    "SBIN",
    "BHARTIARTL",
    "KOTAKBANK",
    "LT",
    "AXISBANK",
    "ASIANPAINT",
    "MARUTI",
    "SUNPHARMA",
    "TITAN",
    "ULTRACEMCO",
    "WIPRO",
    "NESTLEIND",
    "BAJFINANCE",
    "TATAMOTORS",
    "TATASTEEL",
    "POWERGRID",
    "NTPC",
    "ONGC",
    "HCLTECH",
    "JSWSTEEL",
    "GRASIM",
    "CIPLA",
    "COALINDIA"
]

START_DATE = "2018-01-01"
END_DATE   = None          # None = up to the dataset's last session
CACHE      = "data_cache"  # downloaded parquet files land here and are reused

print(len(UNIVERSE), "symbols requested")

In [ ]:
# Ingestion. One small parquet per symbol is pulled from the Hub dataset
# `vishnun0027/indian-market-historical-ohlcv` (2,421 NSE/BSE equities) and
# cleaned. Downloading per symbol rather than snapshotting the repo means a
# 30-name universe fetches 30 small files instead of 283 MB.
#
# Cleaning, in order: back-adjust OHLC by adj_close/close so a split is not read
# as a 90% crash, coerce numerics, drop unparseable dates and missing closes
# (rather than forward-filling, so a gap stays visible), and drop duplicate
# sessions keeping the last.
#
# If the Hub is unreachable the toolkit falls back to a synthetic panel and says
# so loudly. Synthetic results describe the generator, not the market.

panel = L.load_panel(UNIVERSE, start_date=START_DATE, end_date=END_DATE,
                     cache_dir=CACHE)

close = L.align_close_matrix(panel)
print(f"{len(panel)} symbols | {close.index.min().date()} -> {close.index.max().date()}"
      f" | {len(close)} sessions")

In [ ]:
# Features are computed per symbol and left NaN until each window has filled.
# They are never back-filled: a back-filled indicator is a look-ahead, and it is
# invisible in every metric downstream.
feature_panel = L.build_feature_panel(panel)

sample = feature_panel[sorted(feature_panel)[0]]
print(f"{len(sample.columns)} features:", list(sample.columns))
display(sample.dropna().tail(3))

In [ ]:
# Simulation settings, shared by every notebook so the strategies are comparable.
#
# execution_lag=1 is the property that keeps this honest: a signal computed from
# day t's close is traded into day t+1's return. The engine refuses lag=0.
config = L.BacktestConfig(
    initial_capital=1_000_000.0,
    cost_bps=25.0,        # all-in round trip for Indian cash equities
    max_weight=0.10,
    rebalance_days=5,     # weekly; the main control over turnover
    max_gross=1.0,        # long-only, unlevered
    execution_lag=1,
)

benchmark = L.equal_weight_benchmark(close, config)
print("equal-weight buy & hold:",
      {k: round(v, 4) for k, v in benchmark.stats.items()
       if k in ("cagr", "sharpe", "max_drawdown")})

## Signal

In [ ]:
low_volatility_scores = L.low_volatility_scores(feature_panel, top_fraction=0.25)

held = (low_volatility_scores > 0).sum(axis=1)
print(f"names held: mean {held.mean():.1f}")

vol = L.cross_section(feature_panel, "realized_vol_60")
print("\nrealized vol across the universe (annualized):")
display(vol.mean().sort_values().to_frame("mean vol").head(8))

In [ ]:
# The selection is a persistent property, not a fast-moving signal: volatility
# ranks are sticky, which is why this strategy turns over far less than momentum.
import matplotlib.pyplot as plt

ranks = vol.rank(axis=1, pct=True)
fig, ax = plt.subplots(figsize=(11, 4))
for i, symbol in enumerate(sorted(vol.columns)[:8]):
    ax.plot(ranks.index, ranks[symbol], linewidth=1.0,
            color=L.PALETTE[i % len(L.PALETTE)], label=symbol)
ax.axhline(0.25, color="#444444", linestyle="--", linewidth=1, label="selection cut")
ax.set_title("Volatility rank through time (low = selected)")
ax.set_ylabel("cross-sectional percentile")
ax.grid(**L.GRID); ax.legend(ncol=5, fontsize=8)
plt.tight_layout(); plt.show()

## Backtest

Against equal-weight buy-and-hold of the same names — the honest comparison for a long-only stock picker. Beating cash is not the question.

In [ ]:
result = L.run_backtest(low_volatility_scores, close, config)

comparison = L.compare_stats({"low volatility": result, "equal weight": benchmark})
display(comparison)

## Analysis

In [ ]:
L.plot_equity({"low volatility": result}, title="low volatility vs equal weight",
              benchmark=benchmark.returns)

In [ ]:
L.plot_return_profile(result, "low volatility")

In [ ]:
L.plot_exposure(result, "low volatility")

In [ ]:
L.plot_weight_heatmap(result, title="low volatility: allocation over time")

## Does the anomaly show up here?

The claim is that realized volatility is *inversely* related to risk-adjusted
return. Testing it directly on this universe is more informative than the
backtest, because it does not depend on the portfolio construction.

In [ ]:
returns = close.pct_change()
by_symbol = pd.DataFrame({
    "ann_vol": returns.std() * np.sqrt(252),
    "ann_return": (1 + returns).prod() ** (252 / len(returns)) - 1,
})
by_symbol["sharpe"] = by_symbol["ann_return"] / by_symbol["ann_vol"]

import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for ax, column in zip(axes, ("ann_return", "sharpe")):
    ax.scatter(by_symbol["ann_vol"], by_symbol[column], color=L.PALETTE[0], s=30)
    fit = np.polyfit(by_symbol["ann_vol"], by_symbol[column], 1)
    xs = np.linspace(by_symbol["ann_vol"].min(), by_symbol["ann_vol"].max(), 50)
    ax.plot(xs, np.polyval(fit, xs), color=L.PALETTE[1], linestyle="--",
            label=f"slope {fit[0]:+.2f}")
    ax.set_xlabel("annualized volatility"); ax.set_ylabel(column)
    ax.legend(fontsize=8); ax.grid(**L.GRID)
fig.suptitle("Volatility versus outcome, per name", y=1.03)
plt.tight_layout(); plt.show()

print("A negative slope on the right-hand panel is the anomaly. On 30 names over")
print("a few years this is a very weak test — the standard error on that slope is")
print("large enough to admit either sign.")

---

## What this does and does not show

Read before quoting any number above.

- **Survivorship.** The universe is today's large caps, applied to history. Names
  that were large caps in 2018 and are not now are absent, and they are absent
  precisely because they did badly. Every long-only result here is biased upward
  by an amount this notebook cannot measure. A point-in-time constituent list is
  the only fix, and this dataset does not carry one.
- **One universe, one period.** Thirty names over a few years is a single draw.
  The difference between two strategies here is well within what the draw alone
  could produce.
- **Costs are a flat 25 bps.** Real cost scales with size and with how illiquid
  the name is, and the fill is assumed at the close. A strategy whose edge is
  this side of costs is not distinguishable from one that has no edge.
- **No point-in-time fundamentals, no corporate actions beyond the price
  adjustment**, and no circuit-limit modelling. On Indian equities a
  circuit-locked session is untradeable, and the simulation will happily trade it.
- **Parameters were chosen, not fitted.** Nothing here is tuned on a held-out
  period. That is deliberate — tuning on this sample and reporting the result
  would be reporting the tuning.

The purpose of these notebooks is to make the mechanism legible and modifiable,
not to establish that any of these strategies makes money.